In [1]:
# ============================================================
# LUNG SOUND CLASSIFICATION - COMPLETE SYSTEM
# Tüm modelleri config-based test et - Gerçek ses verisi
# ============================================================
import os
import random
import numpy as np
import pandas as pd
import librosa
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Conv1D, MaxPooling1D, Flatten, BatchNormalization, Dropout
from tensorflow.keras.optimizers import Adam, Adamax
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

print("\n" + "="*100)
print("🫁 LUNG SOUND CLASSIFICATION - COMPLETE SYSTEM")
print("="*100)

# ============================================================
# PARAMETRELER
# ============================================================
SEGMENT_SEC = 2.0
SR = 22050
N_MFCC = 40
NUM_CLASSES = 3
EPOCHS = 120
BATCH_SIZE = 32
RANDOM_STATE = 42

# ============================================================
# 1. SES DOSYALARINI YÜKLEME
# ============================================================
print("\n📂 Adım 1: Ses Dosyalarını Yükle")
print("-"*100)

class_folders = {
    "Asthma": "./Datasets/Asthma",
    "COPD": "./Datasets/COPD4",
    "Healthy": "./Datasets/Healthy"
}

# Dosya sayısını kontrol et
total_files = 0
for label, folder in class_folders.items():
    if os.path.exists(folder):
        files = len([f for f in os.listdir(folder) if f.endswith('.wav')])
        print(f"  📁 {label}: {files} dosya")
        total_files += files
    else:
        print(f"  ❌ {label}: Klasör BULUNAMADI - {folder}")

if total_files == 0:
    print("\n❌ HATA: Hiç ses dosyası bulunamadı!")
    print("📂 Lütfen şu yapıyı oluştur:")
    print("   ./Datasets/")
    print("   ├── Asthma/     (*.wav dosyaları)")
    print("   ├── COPD4/      (*.wav dosyaları)")
    print("   └── Healthy/    (*.wav dosyaları)")
    exit()

print(f"\n✅ Toplam {total_files} ses dosyası bulundu")

# ============================================================
# 2. AUDIO UTILS
# ============================================================
def segment_audio(audio, sr, overlap_ratio=0.25):
    """
    Ses dosyasını belirlenen overlap oranıyla segmentlere böler.
    overlap_ratio: 0.20 veya 0.25 gibi bir değer.
    """
    win = int(SEGMENT_SEC * sr)
    # Hop boyutu = win * (1 - overlap_ratio)
    hop = int(win * (1 - overlap_ratio))
    
    return [audio[i:i + win] for i in range(0, len(audio) - win + 1, hop)]

def augment_noise(audio):
    return audio + 0.003 * np.random.randn(len(audio))

def augment_pitch(audio, sr):
    return librosa.effects.pitch_shift(audio, sr=sr, n_steps=random.uniform(-1, 1))

def augment_speed(audio):
    return librosa.effects.time_stretch(audio, rate=random.uniform(0.95, 1.05))

def apply_augmentation(audio, sr):
    return random.choice([
        lambda x: augment_noise(x),
        lambda x: augment_pitch(x, sr),
        lambda x: augment_speed(x)
    ])(audio)

# ============================================================
# 3. FEATURE EXTRACTION
# ============================================================
def extract_mfcc_seq(audio, sr=SR, max_len=100):
    """CNN için MFCC sequence"""
    mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=N_MFCC, n_fft=2048, hop_length=512)
    mfcc = mfcc[:, :max_len]
    if mfcc.shape[1] < max_len:
        mfcc = np.pad(mfcc, ((0,0), (0, max_len - mfcc.shape[1])))
    return mfcc.T

def extract_mfcc_stats(audio, sr=SR):
    """MLP/KAN/SVM için MFCC istatistikleri"""
    mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=N_MFCC, n_fft=2048, hop_length=512)
    delta = librosa.feature.delta(mfcc)
    
    feats = np.hstack([
        mfcc.mean(axis=1),
        mfcc.std(axis=1),
        mfcc.min(axis=1),
        mfcc.max(axis=1),
        delta.mean(axis=1),
        delta.std(axis=1),
    ])
    return feats

# ============================================================
# 4. DATASET LOADING
# ============================================================
print("\n📊 Adım 2: Dataset Yükle ve Feature Extract")
print("-"*100)

X_cnn, X_flat, y = [], [], []

for label, folder in class_folders.items():
    if not os.path.exists(folder):
        continue
    
    files = [f for f in os.listdir(folder) if f.endswith('.wav')]
    print(f"\n  {label}:")
    
    for file_idx, file in enumerate(files):
        try:
            audio, sr = librosa.load(os.path.join(folder, file), sr=SR)
            segments = segment_audio(audio, sr)
            
            for seg in segments:
                # Original segment
                X_cnn.append(extract_mfcc_seq(seg))
                X_flat.append(extract_mfcc_stats(seg))
                y.append(label)
                
                # Augmented segment (1x)
                seg_aug = apply_augmentation(seg, sr)
                X_cnn.append(extract_mfcc_seq(seg_aug))
                X_flat.append(extract_mfcc_stats(seg_aug))
                y.append(label)
            
            if (file_idx + 1) % 5 == 0 or file_idx == len(files) - 1:
                print(f"    ✅ {file_idx + 1}/{len(files)} dosya işlendi")
                
        except Exception as e:
            print(f"    ⚠️  {file}: {e}")

# Convert to numpy
print("\n  Converting to numpy arrays...")
X_cnn = np.array(X_cnn, dtype="float32")
X_flat = np.array(X_flat, dtype="float32")

# Normalize
X_flat = (X_flat - X_flat.mean(axis=0)) / (X_flat.std(axis=0) + 1e-8)

le = LabelEncoder()
y = le.fit_transform(y)

print(f"\n✅ Dataset hazır:")
print(f"   CNN Shape: {X_cnn.shape}")
print(f"   Flat Shape: {X_flat.shape}")
print(f"   Classes: {le.classes_}")
print(f"   Distribution: {np.bincount(y)}")

# ============================================================
# 5. MODEL BUILDERS
# ============================================================
def build_mlp(input_dim, units):
    model = Sequential()
    model.add(Dense(units[0], activation="relu", input_dim=input_dim))
    model.add(BatchNormalization())
    model.add(Dropout(0.3))
    
    for u in units[1:]:
        model.add(Dense(u, activation="relu"))
        model.add(BatchNormalization())
        model.add(Dropout(0.3))
    
    model.add(Dense(NUM_CLASSES, activation="softmax"))
    model.compile(optimizer=Adam(learning_rate=5e-4), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

def build_cnn(input_shape, filters):
    model = Sequential()
    
    for i, f in enumerate(filters):
        kernel_size = 7 if i == 0 else 5 if i == 1 else 3
        model.add(Conv1D(f, kernel_size, padding="same", activation="relu", 
                        input_shape=input_shape if i == 0 else None))
        model.add(BatchNormalization())
        model.add(MaxPooling1D(2))
        model.add(Dropout(0.2))
    
    model.add(Flatten())
    model.add(Dense(128, activation="relu"))
    model.add(BatchNormalization())
    model.add(Dropout(0.3))
    model.add(Dense(NUM_CLASSES, activation="softmax"))
    model.compile(optimizer=Adam(learning_rate=3e-4), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

def build_kan(input_dim, units):
    model = Sequential()
    model.add(Dense(units[0], activation="relu", input_dim=input_dim))
    model.add(BatchNormalization())
    model.add(Dropout(0.3))
    
    for u in units[1:]:
        model.add(Dense(u, activation="relu"))
        model.add(BatchNormalization())
        model.add(Dropout(0.3))
    
    model.add(Dense(NUM_CLASSES, activation="softmax"))
    model.compile(optimizer=Adamax(learning_rate=1e-4), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

# ============================================================
# 6. TRAINING FUNCTION
# ============================================================
def train_and_evaluate(X, y, model_name, config, is_cnn=False):
    """Model eğit ve F1 score döndür"""
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
    )
    
    class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
    class_weight_dict = {i: w for i, w in enumerate(class_weights)}
    
    if is_cnn:
        if len(X_train.shape) == 2:
            X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
            X_test = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)
        model = build_cnn(X_train.shape[1:], config)
    else:
        if model_name == "MLP":
            model = build_mlp(X_train.shape[1], config)
        else:  # KAN
            model = build_kan(X_train.shape[1], config)
    
    early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
    
    model.fit(
        X_train, y_train,
        validation_split=0.15,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        class_weight=class_weight_dict,
        callbacks=[early_stop],
        verbose=0
    )
    
    y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)
    f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
    acc = accuracy_score(y_test, y_pred)
    
    return f1, acc, y_test, y_pred

# ============================================================
# 7. CONFIGS
# ============================================================
configs = {
    "1_layer": [[32], [64], [128], [256]],
    "2_layer": [[32, 64], [64, 128], [128, 256], [256, 512]],
    "3_layer": [[32, 64, 128], [64, 128, 256], [128, 256, 512]],
}

svm_configs = [
    {"C": 0.1, "kernel": "rbf"},
    {"C": 1, "kernel": "rbf"},
    {"C": 5, "kernel": "rbf"},
    {"C": 10, "kernel": "rbf"},
]

# ============================================================
# 8. TESTING - TÜM MODELLERİ TEST ET
# ============================================================
print("\n" + "="*100)
print("🧠 Adım 3: Tüm Modelleri Test Et")
print("="*100)

all_results = {"MLP": {}, "CNN": {}, "KAN": {}, "SVM": {}, "KNN": {}}

# ========== MLP TESTING ==========
print("\n🧠 MLP - LAYER-BY-LAYER CONFIG TESTING")
print("-"*100)

for depth, cfgs in configs.items():
    print(f"  {depth}:")
    for units in cfgs:
        f1, acc, _, _ = train_and_evaluate(X_flat, y, "MLP", units, is_cnn=False)
        all_results["MLP"][str(units)] = f1
        print(f"    {str(units):25s} → F1: {f1:.4f} | Acc: {acc:.4f}")

# ========== CNN TESTING ==========
print("\n🧠 CNN - LAYER-BY-LAYER CONFIG TESTING")
print("-"*100)

for depth, cfgs in configs.items():
    print(f"  {depth}:")
    for units in cfgs:
        f1, acc, _, _ = train_and_evaluate(X_cnn, y, "CNN", units, is_cnn=True)
        all_results["CNN"][str(units)] = f1
        print(f"    {str(units):25s} → F1: {f1:.4f} | Acc: {acc:.4f}")

# ========== KAN TESTING ==========
print("\n🧠 KAN - LAYER-BY-LAYER CONFIG TESTING")
print("-"*100)

for depth, cfgs in configs.items():
    print(f"  {depth}:")
    for units in cfgs:
        f1, acc, _, _ = train_and_evaluate(X_flat, y, "KAN", units, is_cnn=False)
        all_results["KAN"][str(units)] = f1
        print(f"    {str(units):25s} → F1: {f1:.4f} | Acc: {acc:.4f}")

# ========== SVM TESTING ==========
print("\n🧠 SVM - GRID SEARCH")
print("-"*100)

X_train, X_test, y_train, y_test = train_test_split(
    X_flat, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

for cfg in svm_configs:
    svm = SVC(C=cfg["C"], kernel=cfg["kernel"], random_state=RANDOM_STATE)
    svm.fit(X_train, y_train)
    
    y_pred = svm.predict(X_test)
    f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
    acc = accuracy_score(y_test, y_pred)
    
    all_results["SVM"][str(cfg)] = f1
    print(f"  {str(cfg):40s} → F1: {f1:.4f} | Acc: {acc:.4f}")

# ========== KNN TESTING ==========
print("\n🧠 KNN")
print("-"*100)

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)

y_pred = knn.predict(X_test)
f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
acc = accuracy_score(y_test, y_pred)

all_results["KNN"]["n_neighbors=5"] = f1
print(f"  n_neighbors=5 → F1: {f1:.4f} | Acc: {acc:.4f}")

# ============================================================
# 9. FINAL RESULTS
# ============================================================
print("\n" + "="*100)
print("📊 FINAL RESULTS - HER MODELUN BEST CONFIGU")
print("="*100)

best_results = {}
for model_name, configs_dict in all_results.items():
    if configs_dict:
        best_cfg = max(configs_dict.items(), key=lambda x: x[1])
        best_results[model_name] = best_cfg
        print(f"\n{model_name}:")
        print(f"  Config: {best_cfg[0]}")
        print(f"  F1 Score: {best_cfg[1]:.4f}")

# ============================================================
# 10. RANKING
# ============================================================
print("\n" + "="*100)
print("🏆 RANKING - EN İYİ MODELLER")
print("="*100)

ranking = sorted(best_results.items(), key=lambda x: x[1][1], reverse=True)

print("\n{:<10} {:<50} {:<15}".format("Rank", "Model (Config)", "F1 Score"))
print("-"*100)

for rank, (model_name, (config, f1)) in enumerate(ranking, 1):
    print("{:<10} {:<50} {:<15.4f}".format(f"{rank}.", f"{model_name} - {config[:48]}", f1))

# Overall best
overall_best = ranking[0]
print("\n" + "="*100)
print(f"🎯 BEST MODEL: {overall_best[0]}")
print(f"   Config: {overall_best[1][0]}")
print(f"   F1 Score: {overall_best[1][1]:.4f}")
print("="*100)

# ============================================================
# 11. SAVE RESULTS TO CSV
# ============================================================
print("\n💾 CSV Dosyasına Kaydet...")

results_df = []
for model_name, configs_dict in all_results.items():
    for config, f1 in configs_dict.items():
        results_df.append({"Model": model_name, "Config": config, "F1_Score": f1})

df = pd.DataFrame(results_df)
df = df.sort_values("F1_Score", ascending=False)
df.to_csv("lung_sound_results.csv", index=False)

print(f"✅ Sonuçlar kaydedildi: lung_sound_results.csv")
print(f"   ({len(df)} model configuration test edildi)")

# ============================================================
# 12. SUMMARY
# ============================================================
print("\n" + "="*100)
print("✅ TAMAMLANDI!")
print("="*100)

print("\n📊 ÖZET:")
print(f"  • Toplam ses dosyası: {total_files}")
print(f"  • Toplam sample (segment × augmentation): {len(y)}")
print(f"  • Test edilen model configurasyonu: {len(df)}")
print(f"  • En iyi model: {overall_best[0]} (F1: {overall_best[1][1]:.4f})")

print("\n📁 ÇIKTILARı:")
print(f"  • CSV: lung_sound_results.csv")
print(f"  • Konsol: Yukarıdaki tüm sonuçlar")

print("\n💡 SONRAKI ADIM:")
print("  1. lung_sound_results.csv'yi aç (Excel/Sheets)")
print("  2. En iyi modeli seç")
print("  3. Deployment için kullan")

print("\n" + "="*100 + "\n")


🫁 LUNG SOUND CLASSIFICATION - COMPLETE SYSTEM

📂 Adım 1: Ses Dosyalarını Yükle
----------------------------------------------------------------------------------------------------
  📁 Asthma: 96 dosya
  📁 COPD: 112 dosya
  📁 Healthy: 112 dosya

✅ Toplam 320 ses dosyası bulundu

📊 Adım 2: Dataset Yükle ve Feature Extract
----------------------------------------------------------------------------------------------------

  Asthma:
    ✅ 5/96 dosya işlendi
    ✅ 10/96 dosya işlendi
    ✅ 15/96 dosya işlendi
    ✅ 20/96 dosya işlendi
    ✅ 25/96 dosya işlendi
    ✅ 30/96 dosya işlendi
    ✅ 35/96 dosya işlendi
    ✅ 40/96 dosya işlendi
    ✅ 45/96 dosya işlendi
    ✅ 50/96 dosya işlendi
    ✅ 55/96 dosya işlendi
    ✅ 60/96 dosya işlendi
    ✅ 65/96 dosya işlendi
    ✅ 70/96 dosya işlendi
    ✅ 75/96 dosya işlendi
    ✅ 80/96 dosya işlendi
    ✅ 85/96 dosya işlendi
    ✅ 90/96 dosya işlendi
    ✅ 95/96 dosya işlendi
    ✅ 96/96 dosya işlendi

  COPD:
    ✅ 5/112 dosya işlendi
    ✅ 10/11